# 1. Импорты и загрузка данных

In [34]:
import pandas as pd
import numpy as np
from scipy.stats import normaltest, pearsonr
import optuna
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('repo_health_report.csv')
# df.set_index(['repo.platform_id', 'repo.owner', 'repo.name'], inplace=True)
df.shape

(27761, 102)

# 2. Базовая предобработка и заполнение пропусков

In [35]:
bool_cols = [
    'documentation.has_readme', 'documentation.has_license', 'documentation.has_contributing',
    'documentation.has_code_of_conduct', 'documentation.has_changelog', 'documentation.has_docs_dir',
    'documentation.has_issue_templates', 'documentation.has_pr_template', 'cicd.has_ci_config',
    'cicd.has_test_stage', 'cicd.has_lint_or_security_stage', 'cicd.has_deploy_stage',
    'cicd.pipeline_history_available', 'cicd.branch_protection_enabled', 'security.appsec_available',
    'security.has_security_policy', 'activity.releases.uses_semver'
]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].fillna(False)

zero_cols = [
    'documentation.docs_file_count', 'documentation.readme_length_chars', 'cicd.declared_stage_count',
    'cicd.runs_30d', 'security.open_defect_groups_total', 'security.resolved_false_positive_total',
    'activity.commits_30d', 'activity.commits_90d', 'activity.commits_365d', 'activity.contributors_365d',
    'activity.branch_count', 'activity.tag_count', 'activity.pull_requests.open', 'activity.pull_requests.merged_90d',
    'activity.pull_requests.stale_open_count', 'activity.releases.count', 'activity.likes.value',
    'activity.likes.percentile', 'issues.open_count', 'issues.opened_90d', 'issues.closed_90d',
    'issues.stale_open_count', 'issues.unanswered_open_count', 'code_health.file_count',
    'code_health.lines_of_code_estimate', 'code_health.todo_count', 'code_health.fixme_count',
    'code_health.hack_count', 'code_health.largest_file_lines'
]
for col in zero_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

time_and_rate_cols = [
    'cicd.median_duration_seconds', 'security.oldest_open_defect_group_age_days',
    'activity.pull_requests.avg_merge_time_hours', 'activity.releases.median_interval_days',
    'issues.avg_time_to_first_response_hours', 'issues.avg_time_to_close_hours',
    'code_health.oldest_todo_age_days', 'cicd.success_rate_30d', 'activity.bus_factor_top1_share_365d',
    'code_health.todo_density_per_kloc'
]
for col in time_and_rate_cols:
    if col in df.columns:
        df[col] = df[col].fillna(-1.0)

date_cols = [
    'documentation.last_readme_change_at', 'cicd.last_run_at', 'security.latest_scan',
    'activity.last_commit_at', 'activity.releases.last_release_at'
]

CURRENT_DATE = pd.to_datetime(df['collection.collected_at'], utc=True).fillna(pd.Timestamp.now(tz='UTC'))
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
        df[f'{col}_days_ago'] = (CURRENT_DATE - df[col]).dt.days
        df[f'{col}_days_ago'] = df[f'{col}_days_ago'].fillna(-1.0)

date_cols_days_ago = [f'{c}_days_ago' for c in date_cols]

cat_cols = ['documentation.license_type', 'cicd.last_run_status', 'repo.primary_language']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')
        

# 3. Нормализация данных и расчет Score-колонок

In [36]:
heavy_log_cols = [
    'cicd.median_duration_seconds', 'security.oldest_open_defect_group_age_days',
    'activity.pull_requests.avg_merge_time_hours', 'activity.releases.median_interval_days',
    'issues.avg_time_to_first_response_hours', 'issues.avg_time_to_close_hours',
    'code_health.oldest_todo_age_days', 'documentation.readme_length_chars',
    'code_health.lines_of_code_estimate', 'code_health.largest_file_lines',
    'activity.commits_365d', 'activity.likes.value', 'activity.contributors_365d', 'issues.open_count'
]
for col in heavy_log_cols:
    if col in df.columns:
        valid_mask = df[col] != -1.0
        df.loc[valid_mask, col] = np.log1p(df.loc[valid_mask, col])

for col in bool_cols:
    if col in df.columns:
        df[col + '_score'] = df[col].astype(int) * 100

if 'cicd.success_rate_30d' in df.columns:
    valid_sr = df['cicd.success_rate_30d'] != -1.0
    df['cicd.success_rate_30d_score'] = 50.0
    df.loc[valid_sr, 'cicd.success_rate_30d_score'] = df.loc[valid_sr, 'cicd.success_rate_30d'] * 100

if 'activity.bus_factor_top1_share_365d' in df.columns:
    valid_bf = df['activity.bus_factor_top1_share_365d'] != -1.0
    df['activity.bus_factor_top1_share_365d_score'] = 50.0
    df.loc[valid_bf, 'activity.bus_factor_top1_share_365d_score'] = (1.0 - df.loc[valid_bf, 'activity.bus_factor_top1_share_365d']) * 100


In [37]:
status_map = {'SUCCESS': 100, 'PASSED': 100, 'FAILED': 0, 'ERROR': 0}
if 'cicd.last_run_status' in df.columns:
    df['cicd.last_run_status_score'] = df['cicd.last_run_status'].map(lambda x: status_map.get(x, 50))

if 'documentation.license_type' in df.columns:
    df['documentation.license_type_score'] = df['documentation.license_type'].map(lambda x: 100 if x in ['BSD', 'MIT', 'Apache-2.0', 'GPL'] else 50)
    

In [38]:
def normalize_metric_safe(series, missing_val=-1.0, higher_is_better=True, neutral_score=50.0):
    scores = series.copy().astype(float)
    valid = scores != missing_val

    if not valid.any():
        return pd.Series(neutral_score, index=series.index)

    min_val = scores[valid].min()
    max_val = scores[valid].max()

    if max_val == min_val:
        scores.loc[valid] = 100.0 if higher_is_better else neutral_score
    else:
        if higher_is_better:
            scaled = ((scores[valid] - min_val) / (max_val - min_val)) * 100
        else:
            scaled = ((max_val - scores[valid]) / (max_val - min_val)) * 100
        scores.loc[valid] = np.clip(scaled, 0, 100)

    scores.loc[~valid] = neutral_score
    return scores

higher_is_better_cols = [
    'documentation.docs_file_count', 'documentation.readme_length_chars', 'cicd.declared_stage_count',
    'cicd.runs_30d', 'security.resolved_false_positive_total', 'activity.commits_30d', 'activity.commits_90d',
    'activity.commits_365d', 'activity.contributors_365d', 'activity.branch_count', 'activity.tag_count',
    'activity.pull_requests.merged_90d', 'activity.releases.count', 'activity.likes.value',
    'activity.likes.percentile', 'issues.closed_90d', 'code_health.file_count', 'code_health.lines_of_code_estimate'
]
lower_is_better_cols = [
    'documentation.last_readme_change_at_days_ago', 'cicd.last_run_at_days_ago', 'cicd.median_duration_seconds',
    'security.open_defect_groups_total', 'security.oldest_open_defect_group_age_days', 'security.latest_scan_days_ago',
    'activity.last_commit_at_days_ago', 'activity.pull_requests.open', 'activity.pull_requests.stale_open_count',
    'activity.pull_requests.avg_merge_time_hours', 'activity.releases.median_interval_days',
    'activity.releases.last_release_at_days_ago', 'issues.open_count', 'issues.opened_90d',
    'issues.stale_open_count', 'issues.unanswered_open_count', 'issues.avg_time_to_first_response_hours',
    'issues.avg_time_to_close_hours', 'code_health.largest_file_lines', 'code_health.todo_count',
    'code_health.fixme_count', 'code_health.hack_count', 'code_health.todo_density_per_kloc',
    'code_health.oldest_todo_age_days'
]

for col in higher_is_better_cols:
    if col in df.columns:
        df[col + '_score'] = normalize_metric_safe(df[col], higher_is_better=True)

for col in lower_is_better_cols:
    if col in df.columns:
        df[col + '_score'] = normalize_metric_safe(df[col], higher_is_better=False)
        

In [39]:
def map_severity_score(val):
    if pd.isna(val) or val == 'Unknown': return 50.0
    text = str(val).upper()
    if 'CRITICAL' in text or 'HIGH' in text: return 0.0
    if 'MEDIUM' in text: return 60.0
    if 'LOW' in text: return 80.0
    return 50.0

def map_status_score(val):
    if pd.isna(val) or val == 'Unknown': return 50.0
    text = str(val).upper()
    if 'RESOLVED' in text or 'CLOSED' in text: return 100.0
    if 'OPEN' in text: return 20.0
    return 50.0

def map_engine_score(val):
    if pd.isna(val) or val == 'Unknown': return 50.0
    return 100.0 if len(str(val)) > 0 else 50.0


if 'security.defect_groups_by_severity' in df.columns:
    df['security.defect_groups_by_severity_score'] = df['security.defect_groups_by_severity'].apply(map_severity_score)
if 'security.defect_groups_by_status' in df.columns:
    df['security.defect_groups_by_status_score'] = df['security.defect_groups_by_status'].apply(map_status_score)
if 'security.defect_groups_by_engine_type' in df.columns:
    df['security.defect_groups_by_engine_type_score'] = df['security.defect_groups_by_engine_type'].apply(map_engine_score)
    

# 4. Сборка категорий и Optuna

In [40]:
def aggregate_category_scores(df):
    categories = {
        'score_security': 'security.',
        'score_cicd': 'cicd.',
        'score_issues': 'issues.',
        'score_activity': 'activity.',
        'score_docs': 'documentation.',
        'score_health': 'code_health.'
    }

    df_agg = pd.DataFrame(index=df.index)
    for cat_score, prefix in categories.items():
        cols = [c for c in df.columns if c.startswith(prefix) and c.endswith('_score')]
        if cols:
            df_agg[cat_score] = df[cols].mean(axis=1).fillna(50.0)
        else:
            df_agg[cat_score] = 50.0
    return df_agg

df_categories = aggregate_category_scores(df)
df = pd.concat([df, df_categories], axis=1)

if 'collection.category_status.security' in df.columns:
    df['is_security_active'] = (df['collection.category_status.security'] != 'unavailable').astype(int)
else:
    df['is_security_active'] = 1

if 'collection.category_status.cicd' in df.columns:
    df['is_cicd_active'] = (df['collection.category_status.cicd'] != 'unavailable').astype(int)
else:
    df['is_cicd_active'] = 1

df['is_issues_active'] = 1
df['is_activity_active'] = 1
df['is_docs_active'] = 1
df['is_health_active'] = 1


In [41]:
def objective(trial, df):
    # Подбор весов
    w_sec = trial.suggest_float('w_sec', 0.15, 0.30)
    w_health = trial.suggest_float('w_health', 0.15, 0.30)
    w_issues = trial.suggest_float('w_issues', 0.10, 0.25)
    w_act = trial.suggest_float('w_act', 0.10, 0.25)
    w_doc = trial.suggest_float('w_doc', 0.10, 0.25)
    w_cicd = trial.suggest_float('w_cicd', 0.10, 0.25)
    
    base_w = [0.20, 0.20, 0.15, 0.15, 0.15, 0.15]
    current_w = [w_sec, w_health, w_issues, w_act, w_doc, w_cicd]
    l2_penalty = sum((c - b) ** 2 for c, b in zip(current_w, base_w))
    
    temp_weights = pd.DataFrame({
        'sec': w_sec * df['is_security_active'],
        'health': w_health * df['is_health_active'],
        'iss': w_issues * df['is_issues_active'],
        'act': w_act * df['is_activity_active'],
        'doc': w_doc * df['is_docs_active'],
        'cicd': w_cicd * df['is_cicd_active']
    }, index=df.index)

    norm_weights = temp_weights.div(temp_weights.sum(axis=1), axis=0)

    health_score = (
        df['score_security'] * norm_weights['sec'] +
        df['score_health'] * norm_weights['health'] +
        df['score_issues'] * norm_weights['iss'] +
        df['score_activity'] * norm_weights['act'] +
        df['score_docs'] * norm_weights['doc'] +
        df['score_cicd'] * norm_weights['cicd']
    )

    mean_score = health_score.mean()
    std_score = health_score.std()
    
    mean_penalty = abs(mean_score - 65.0)
    corr_penalty = 0
    
    if 'activity.likes.value_score' in df.columns:
        likes_score = df['activity.likes.value_score']
        valid_mask = ~(health_score.isna() | likes_score.isna())
        if valid_mask.sum() > 1:
            corr, _ = pearsonr(health_score[valid_mask], likes_score[valid_mask])
            corr_penalty = abs(corr - 0.3) * 100
    
    loss = mean_penalty + corr_penalty - (std_score * 0.5) + (l2_penalty * 10)
    return loss

sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='minimize', sampler=sampler)

study.optimize(lambda trial: objective(trial, df), n_trials=500)

print("Лучшие веса:", study.best_params)


[I 2026-09-22 22:32:30,622] A new study created in memory with name: no-name-a5c6e941-eff4-4faa-94de-a9da110ed303
[I 2026-09-22 22:32:30,629] Trial 0 finished with value: 35.1677763478198 and parameters: {'w_sec': 0.20618101782710435, 'w_health': 0.2926071459614874, 'w_issues': 0.20979909127171076, 'w_act': 0.1897987726295555, 'w_doc': 0.12340279606636548, 'w_cicd': 0.12339917805043041}. Best is trial 0 with value: 35.1677763478198.
[I 2026-09-22 22:32:30,636] Trial 1 finished with value: 37.63867481042533 and parameters: {'w_sec': 0.15871254182522992, 'w_health': 0.2799264218662403, 'w_issues': 0.1901672517614813, 'w_act': 0.20621088666940685, 'w_doc': 0.10308767414437037, 'w_cicd': 0.24548647782429914}. Best is trial 0 with value: 35.1677763478198.
[I 2026-09-22 22:32:30,642] Trial 2 finished with value: 40.27861811885129 and parameters: {'w_sec': 0.27486639612006325, 'w_health': 0.1818508666017414, 'w_issues': 0.1272737450810651, 'w_act': 0.1275106764780151, 'w_doc': 0.1456363364439

Лучшие веса: {'w_sec': 0.19825120164613633, 'w_health': 0.2984603610303077, 'w_issues': 0.19069167060059813, 'w_act': 0.10818915992051391, 'w_doc': 0.1003632490191065, 'w_cicd': 0.10162157444677651}


# 5. Создание итогового рейтинга

In [42]:
BEST_WEIGHTS = study.best_params

dynamic_w = pd.DataFrame({
    'sec': BEST_WEIGHTS['w_sec'] * df['is_security_active'],
    'health': BEST_WEIGHTS['w_health'] * df['is_health_active'],
    'iss': BEST_WEIGHTS['w_issues'] * df['is_issues_active'],
    'act': BEST_WEIGHTS['w_act'] * df['is_activity_active'],
    'doc': BEST_WEIGHTS['w_doc'] * df['is_docs_active'],
    'cicd': BEST_WEIGHTS['w_cicd'] * df['is_cicd_active']
}, index=df.index)

norm_w = dynamic_w.div(dynamic_w.sum(axis=1), axis=0)

df['total_health_score'] = (
    df['score_security'] * norm_w['sec'] +
    df['score_health'] * norm_w['health'] +
    df['score_issues'] * norm_w['iss'] +
    df['score_activity'] * norm_w['act'] +
    df['score_docs'] * norm_w['doc'] +
    df['score_cicd'] * norm_w['cicd']
)

leaderboard_cols = [
    'score_security', 'score_health', 'score_issues', 
    'score_activity', 'score_docs', 'score_cicd', 
    'total_health_score'
]

df_leaderboard = df[leaderboard_cols].round(1).copy()
df_leaderboard = df_leaderboard.sort_values(by='total_health_score', ascending=False)

df_leaderboard.head()

,score_security,score_health,score_issues,score_activity,score_docs,score_cicd,total_health_score
13735,55.6,68.2,71.4,35.1,89.0,51.4,65.0
17343,44.4,66.2,71.4,35.9,90.8,53.5,64.8
21776,55.6,69.7,71.4,38.4,80.3,50.7,64.8
24278,44.4,70.0,71.4,38.0,67.4,61.8,64.6
23870,44.4,72.3,78.7,49.4,34.4,60.4,64.5


# 6. Рекомендации

In [43]:
class SourceCraftRecommendationEngine:
    def __init__(self):
        # Приоритизированный список: от Critical до Low
        self.rules = [
            # ================= 1. SECURITY =================
            {
                "category": "Security",
                "priority": "Критический",
                "condition": lambda r: str(r.get('security.defect_groups_by_severity', '')).upper() in ['CRITICAL', 'HIGH'],
                "problem": "AppSec SourceCraft обнаружил уязвимости со статусом CRITICAL или HIGH.",
                "importance": "Уязвимости в коде или зависимостях ведут к прямому риску взлома.",
                "action": "Срочно изучите отчет AppSec и обновите уязвимые пакеты.",
                "impact": "Устранение критических дефектов вернет основные баллы в категории Security."
            },
            {
                "category": "Security",
                "priority": "Высокий",
                "condition": lambda r: r.get('security.oldest_open_defect_group_age_days', 0) > 30,
                "problem": "Обнаружены открытые дефекты безопасности старше 30 дней.",
                "importance": "Игнорирование уязвимостей увеличивает окно возможностей для атак.",
                "action": "Проведите триаж старых уязвимостей: закройте их или отметьте как ложные срабатывания (RESOLVED_FP).",
                "impact": "Снизит штраф за возраст открытых дефектов."
            },
            {
                "category": "Security",
                "priority": "Средний",
                "condition": lambda r: r.get('security.has_security_policy', False) == False,
                "problem": "Отсутствует файл политики безопасности (SECURITY.md).",
                "importance": "Пользователи не знают, как безопасно сообщить вам о найденных багах.",
                "action": "Добавьте SECURITY.md с инструкцией по репорту уязвимостей.",
                "impact": "Повысит оценку применения практик безопасной разработки."
            },

            # ================= 2. CI/CD =================
            {
                "category": "CI/CD",
                "priority": "Высокий",
                "condition": lambda r: r.get('cicd.has_ci_config', False) == False,
                "problem": "В проекте отсутствует конфигурация CI-пайплайна.",
                "importance": "Без настроенного CI невозможно автоматизировать сборку и тестирование.",
                "action": "Добавьте конфигурационный файл пайплайна (например, .gitlab-ci.yml).",
                "impact": "Разблокирует категорию CI/CD для начисления баллов."
            },
            {
                "category": "CI/CD",
                "priority": "Высокий",
                "condition": lambda r: r.get('cicd.branch_protection_enabled', False) == False,
                "problem": "Защита основных веток (Branch Protection) отключена.",
                "importance": "Любой разработчик может запушить сломанный код напрямую в default-ветку.",
                "action": "Включите защиту веток в настройках репозитория (требование code review).",
                "impact": "Существенно улучшит оценку надежности CI/CD."
            },
            {
                "category": "CI/CD",
                "priority": "Средний",
                "condition": lambda r: r.get('cicd.success_rate_30d', 1.0) != -1.0 and r.get('cicd.success_rate_30d', 1.0) < 0.8,
                "problem": "Доля успешных пайплайнов за последние 30 дней ниже 80%.",
                "importance": "Нестабильные сборки (flaky tests, ошибки среды) блокируют доставку новых фичей.",
                "action": "Стабилизируйте падающие шаги пайплайна и исправьте нестабильные тесты.",
                "impact": "Напрямую повысит метрику cicd.success_rate_30d_score."
            },
            {
                "category": "CI/CD",
                "priority": "Средний",
                "condition": lambda r: r.get('cicd.has_test_stage', False) == False and r.get('cicd.has_ci_config', False) == True,
                "problem": "В CI-пайплайне не настроен этап тестирования (test stage).",
                "importance": "Сборка кода без прогона автотестов не гарантирует его работоспособность.",
                "action": "Добавьте шаг запуска unit/интеграционных тестов в ваш CI-конфиг.",
                "impact": "Повысит балл за полноту конвейера."
            },

            # ================= 3. DOCUMENTATION =================
            {
                "category": "Documentation",
                "priority": "Высокий",
                "condition": lambda r: r.get('documentation.has_readme', False) == False,
                "problem": "Отсутствует файл README.",
                "importance": "README — точка входа для понимания архитектуры, локального запуска и сборки проекта.",
                "action": "Создайте README.md с подробным описанием.",
                "impact": "Значительно повысит оценку в категории документации."
            },
            {
                "category": "Documentation",
                "priority": "Низкий",
                "condition": lambda r: r.get('documentation.has_readme', False) == True and r.get('documentation.readme_length_chars', 1000) < 300,
                "problem": "Файл README слишком короткий (менее 300 символов).",
                "importance": "Короткий README обычно не содержит инструкций по сборке и запуску.",
                "action": "Дополните README разделами 'Getting Started', 'Prerequisites' и 'Installation'.",
                "impact": "Повысит качество оценки документации."
            },
            {
                "category": "Documentation",
                "priority": "Высокий",
                "condition": lambda r: r.get('documentation.has_license', False) == False,
                "problem": "В проекте не указана лицензия.",
                "importance": "Без явной лицензии открытый код не может легально использоваться другими командами.",
                "action": "Добавьте файл LICENSE (например, Apache-2.0 или MIT).",
                "impact": "Улучшит показатель применения лучших практик (Best Practices)."
            },
            {
                "category": "Documentation",
                "priority": "Средний",
                "condition": lambda r: r.get('documentation.has_contributing', False) == False,
                "problem": "Отсутствует файл CONTRIBUTING.",
                "importance": "Потенциальные контрибьюторы не знают ваших правил написания кода и создания PR.",
                "action": "Добавьте файл CONTRIBUTING.md с правилами работы над проектом.",
                "impact": "Улучшит скоринг по стандартам Open Source."
            },

            # ================= 4. ISSUES =================
            {
                "category": "Issues",
                "priority": "Высокий",
                "condition": lambda r: r.get('issues.unanswered_open_count', 0) > 3,
                "problem": "Обнаружены открытые задачи (issues) без ответов мейнтейнеров.",
                "importance": "Игнорирование баг-репортов демотивирует комьюнити и снижает активность.",
                "action": "Проведите первичный триаж: ответьте пользователям, запросите логи или закройте тикеты.",
                "impact": "Улучшит метрики скорости реакции мейнтейнеров."
            },
            {
                "category": "Issues",
                "priority": "Средний",
                "condition": lambda r: r.get('issues.avg_time_to_first_response_hours', 0) > 72,
                "problem": "Среднее время первого ответа на Issue превышает 3 дня (72 часа).",
                "importance": "Медленная реакция замедляет цикл исправления багов.",
                "action": "Настройте систему уведомлений для оперативной реакции на новые Issues.",
                "impact": "Снизит штраф за задержки в обработке багов."
            },
            {
                "category": "Issues",
                "priority": "Низкий",
                "condition": lambda r: r.get('issues.stale_open_count', 0) > 5,
                "problem": "Скопилось много зависших (stale) открытых задач.",
                "importance": "Накопление старых тикетов превращает бэклог в свалку.",
                "action": "Закройте неактуальные задачи или переведите их в статус 'Отложено'.",
                "impact": "Очистит метрику 'stale issues' и повысит Health Score."
            },

            # ================= 5. ACTIVITY =================
            {
                "category": "Activity",
                "priority": "Высокий",
                "condition": lambda r: r.get('activity.bus_factor_top1_share_365d', 0) > 0.85,
                "problem": "Риск Bus Factor: один разработчик вносит более 85% коммитов за год.",
                "importance": "Если ключевой мейнтейнер покинет проект, поддержка полностью остановится.",
                "action": "Распределяйте задачи между участниками команды, привлекайте новых контрибьюторов.",
                "impact": "Улучшит баланс активности и повысит устойчивость репозитория."
            },
            {
                "category": "Activity",
                "priority": "Средний",
                "condition": lambda r: r.get('activity.last_commit_at_days_ago', 0) > 90,
                "problem": "Проект не обновлялся более 3 месяцев.",
                "importance": "Отсутствие активности сигнализирует пользователям о том, что проект заброшен.",
                "action": "Смержите накопившиеся PR или выпустите техническое обновление зависимостей.",
                "impact": "Повысит Health Score активности репозитория."
            },

            # ================= 6. CODE HEALTH =================
            {
                "category": "Code Health",
                "priority": "Средний",
                "condition": lambda r: r.get('code_health.largest_file_lines', 0) > 1500,
                "problem": "В проекте обнаружены огромные монолитные файлы (более 1500 строк).",
                "importance": "Такие файлы крайне тяжело ревьюить, тестировать и поддерживать.",
                "action": "Разбейте логику крупного файла на мелкие модули/классы.",
                "impact": "Снизит штраф за технический долг и сложность."
            },
            {
                "category": "Code Health",
                "priority": "Низкий",
                "condition": lambda r: r.get('code_health.oldest_todo_age_days', 0) > 180,
                "problem": "В коде присутствуют комментарии TODO/FIXME старше полугода.",
                "importance": "Забытые TODO свидетельствуют о неконтролируемом росте технического долга.",
                "action": "Удалите неактуальные TODO, а требующие внимания перенесите в трекер задач.",
                "impact": "Улучшит метрики чистоты кода (Maintainability)."
            },
            {
                "category": "Code Health",
                "priority": "Высокий",
                "condition": lambda r: r.get('code_health.hack_count', 0) > 0,
                "problem": "В коде обнаружены маркеры костылей (HACK).",
                "importance": "HACK-решения хрупкие и часто ломаются при обновлениях.",
                "action": "Проведите рефакторинг участков кода, отмеченных как HACK.",
                "impact": "Повысит оценку надежности кодовой базы."
            }
        ]

    def generate_recommendations(self, row_data):
        active_recs = []
        for rule in self.rules:
            try:
                if rule["condition"](row_data):
                    clean_rule = {
                        "category": rule["category"],
                        "priority": rule["priority"],
                        "problem": rule["problem"],
                        "importance": rule["importance"],
                        "action": rule["action"],
                        "impact": rule["impact"]
                    }
                    
                    # Динамическая подстановка значений
                    if "зависших (stale)" in clean_rule["problem"] and 'issues.stale_open_count' in row_data:
                        clean_rule["problem"] = f"Обнаружено {int(row_data['issues.stale_open_count'])} зависших (stale) открытых задач."
                    
                    if "без ответов" in clean_rule["problem"] and 'issues.unanswered_open_count' in row_data:
                        clean_rule["problem"] = f"Обнаружено {int(row_data['issues.unanswered_open_count'])} открытых задач без ответов мейнтейнеров."
                        
                    if "Bus Factor" in clean_rule["problem"] and 'activity.bus_factor_top1_share_365d' in row_data:
                        share = round(row_data['activity.bus_factor_top1_share_365d'] * 100)
                        clean_rule["problem"] = f"Риск Bus Factor: один разработчик вносит {share}% коммитов за год."
                        
                    active_recs.append(clean_rule)
            except Exception:
                pass
                
        return active_recs

In [44]:
engine = SourceCraftRecommendationEngine()

recommendations = engine.generate_recommendations(df.iloc[1])

print(f"Найдено рекомендаций: {len(recommendations)}\n")

for i, rec in enumerate(recommendations, 1):
    print(f"--- Рекомендация #{i} [{rec['priority']}] ---")
    print(f"Категория: {rec['category']}")
    print(f"Проблема:  {rec['problem']}")
    print(f"Важность:  {rec['importance']}")
    print(f"Действие:  {rec['action']}")
    print(f"Влияние:   {rec['impact']}\n")

Найдено рекомендаций: 6

--- Рекомендация #1 [Средний] ---
Категория: Security
Проблема:  Отсутствует файл политики безопасности (SECURITY.md).
Важность:  Пользователи не знают, как безопасно сообщить вам о найденных багах.
Действие:  Добавьте SECURITY.md с инструкцией по репорту уязвимостей.
Влияние:   Повысит оценку применения практик безопасной разработки.

--- Рекомендация #2 [Высокий] ---
Категория: CI/CD
Проблема:  Защита основных веток (Branch Protection) отключена.
Важность:  Любой разработчик может запушить сломанный код напрямую в default-ветку.
Действие:  Включите защиту веток в настройках репозитория (требование code review).
Влияние:   Существенно улучшит оценку надежности CI/CD.

--- Рекомендация #3 [Низкий] ---
Категория: Documentation
Проблема:  Файл README слишком короткий (менее 300 символов).
Важность:  Короткий README обычно не содержит инструкций по сборке и запуску.
Действие:  Дополните README разделами 'Getting Started', 'Prerequisites' и 'Installation'.
Влияние:

In [45]:
study.best_params

{'w_sec': 0.19825120164613633,
 'w_health': 0.2984603610303077,
 'w_issues': 0.19069167060059813,
 'w_act': 0.10818915992051391,
 'w_doc': 0.1003632490191065,
 'w_cicd': 0.10162157444677651}